In [ ]:
#Importing dependencies
import pandas as pd
from google.colab import files


ModuleNotFoundError: No module named 'google'

In [ ]:
# Load raw tables
sales = pd.read_csv('https://drive.google.com/uc?export=download&id=1vrypPferlfcZFJTTBKExYAvLxvat6w0Z')
customers = pd.read_csv('https://drive.google.com/uc?export=download&id=16iirMkTGEWdqge_tNs18hkBY-hBrigcD')
products = pd.read_csv('https://drive.google.com/uc?export=download&id=1hFeWT03Cr6EImUzO-bTJ-i2d7q3sp83Z')
stores = pd.read_csv('https://drive.google.com/uc?export=download&id=1cGYO7XGxZd11lz7m8PxiRKssYgkwJGtW')
calendar = pd.read_csv('https://drive.google.com/uc?export=download&id=1jnbXUjOaWxb6oTQh7ZGc8hrsvRbBTZzz')


In [ ]:
dfs = {
    "sales": sales,
    "customers": customers,
    "products": products,
    "stores": stores,
    "calendar": calendar
}

# Fast null scan across all source tables
null_summary = {name: df.isnull().sum() for name, df in dfs.items()}

for name, summary in null_summary.items():
    print(f"\n{name.upper()}")
    print(summary)



In [ ]:
# Duplicate check
# Sales uses full-row dupes
{
    "sales": sales.duplicated().sum(),
    "customers": customers.duplicated(subset=['customer_id']).sum(),
    "products": products.duplicated(subset=['product_id']).sum(),
    "stores": stores.duplicated(subset=['store_id']).sum(),
    "calendar": calendar.duplicated(subset=['date']).sum()
}

In [ ]:
# These fields are required for downstream joins
sales = sales.dropna(subset=[
    'customer_id',
    'product_id',
    'store_id',
    'order_date'
])

In [ ]:
# Coerce bad date strings, then drop them
sales['order_date'] = pd.to_datetime(sales['order_date'], errors='coerce')
sales = sales.dropna(subset=['order_date'])

In [ ]:
# Parse remaining date fields
customers['join_date'] = pd.to_datetime(customers['join_date'], errors='coerce')

In [ ]:
# Parse and clean calendar date
calendar['date'] = pd.to_datetime(calendar['date'], errors='coerce')
calendar = calendar.dropna(subset=['date'])

In [ ]:
# Check referential gaps before merge
print("Missing customers:",
      sales[~sales['customer_id'].isin(customers['customer_id'])].shape[0])

print("Missing products:",
      sales[~sales['product_id'].isin(products['product_id'])].shape[0])

print("Missing stores:",
      sales[~sales['store_id'].isin(stores['store_id'])].shape[0])

In [ ]:
# Strip whitespace so ID matching is reliable
sales['product_id'] = sales['product_id'].astype(str).str.strip()
products['product_id'] = products['product_id'].astype(str).str.strip()

In [ ]:
# Inspect missing product IDs before patching
missing_products = sales[~sales['product_id'].isin(products['product_id'])]

missing_products['product_id'].value_counts().head(20)

In [ ]:
# Patch missing product IDs with placeholders
missing_ids = ['P0000', 'P0201']

new_rows = pd.DataFrame({
    'product_id': missing_ids,
    'product_name': ['Unknown Product', 'Unknown Product'],
    'brand': ['Unknown Brand', 'Unknown Brand'],
    'category': ['Unknown', 'Unknown'],
    'cocoa_percent': [0, 0],
    'weight_g': [products['weight_g'].median(), products['weight_g'].median()]
})

products = pd.concat([products, new_rows], ignore_index=True)

In [ ]:
print(
    sales[~sales['product_id'].isin(products['product_id'])].shape[0]
)

In [ ]:
# Normalize key formatting before merges
sales['customer_id'] = sales['customer_id'].astype(str).str.strip()
sales['product_id'] = sales['product_id'].astype(str).str.strip()
sales['store_id'] = sales['store_id'].astype(str).str.strip()

customers['customer_id'] = customers['customer_id'].astype(str).str.strip()
products['product_id'] = products['product_id'].astype(str).str.strip()
stores['store_id'] = stores['store_id'].astype(str).str.strip()

In [ ]:
# Build final merged dataset
df = sales.merge(customers, on='customer_id', how='left')

In [ ]:
df = df.merge(products, on='product_id', how='left')

In [ ]:
df = df.merge(stores, on='store_id', how='left')

In [ ]:
df = df.merge(calendar, left_on='order_date', right_on='date', how='left')

In [ ]:
# Drop helper column from calendar join
df = df.drop(columns=['date'])

In [ ]:
# Final sanity check
print(df.isnull().sum())
print(df.shape)

In [ ]:
# Save merged output
df.to_csv("final_merged_dataset.csv", index=False)

In [ ]:
# Download CSV in Colab
files.download("final_merged_dataset.csv")